# N065 · 回溯去重与剪枝

**目标：** 区分同层重复和同一路径重复使用。

**先修：** N064, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排序；起点；used标记；正数剪枝前提；状态恢复。

## 从问题到算法

值重复时，“索引不同”不再意味着答案不同。排序后在同一层跳过相同值，防止不同兄弟分支生成同一答案；排列还要保证相同值按固定索引顺序使用。组合和允许同一候选多次使用，但必须要求候选严格正，否则递归可能不终止。

## 最小实现

**本章接口：** `subsets_with_dup(nums)`、`permute_unique(nums)`、`combination_sum(candidates, target)`

In [1]:
def subsets_with_dup(nums):
    a=sorted(nums); out=[]; path=[]
    def visit(start):
        out.append(path[:])
        for i in range(start,len(a)):
            if i>start and a[i]==a[i-1]: continue
            path.append(a[i]); visit(i+1); path.pop()
    visit(0); return out

def permute_unique(nums):
    a=sorted(nums); used=[False]*len(a); out=[]; path=[]
    def visit():
        if len(path)==len(a): out.append(path[:]); return
        for i,x in enumerate(a):
            if used[i] or (i>0 and x==a[i-1] and not used[i-1]): continue
            used[i]=True; path.append(x); visit(); path.pop(); used[i]=False
    visit(); return out

def combination_sum(candidates,target):
    a=sorted(set(candidates))
    if any(x<=0 for x in a) or target<0: raise ValueError('positive candidates and nonnegative target required')
    out=[]; path=[]
    def visit(start,remaining):
        if remaining==0: out.append(path[:]); return
        for i in range(start,len(a)):
            if a[i]>remaining: break
            path.append(a[i]); visit(i,remaining-a[i]); path.pop()
    visit(0,target); return out

## 正确性、前提与复杂度

同层相同值的分支在剩余可选值上等价，保留一个代表不会漏掉不同值答案；不同深度可继续使用同一值。正候选使remaining严格下降，保证终止。

**代价：** 输出敏感的指数算法；排序O(n log n)，工作空间为最大递归深度，组合和深度最多target/min(candidates)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,2]
show_table(['对象','去重答案'],[('子集',subsets_with_dup(a)),('排列',permute_unique(a)),('组合和 target=5',combination_sum([1,2],5))])

对象,去重答案
子集,"[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]"
排列,"[[1, 2, 2], [2, 1, 2], [2, 2, 1]]"
组合和 target=5,"[[1, 1, 1, 1, 1], [1, 1, 1, 2], [1, 2, 2]]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import combinations,permutations,product
assert len(subsets_with_dup([1,2,2]))==6
assert len(permute_unique([1,1,2]))==3
assert combination_sum([2,3],0)==[[]]
for a in product([1,2],repeat=5):
    out=subsets_with_dup(a); ref={tuple(sorted(t)) for k in range(6) for t in combinations(a,k)}
    assert set(map(tuple,out))==ref and len(out)==len(ref)
    out=permute_unique(a); ref=set(permutations(a))
    assert set(map(tuple,out))==ref and len(out)==len(ref)
assert {tuple(x) for x in combination_sum([2,3,6,7],7)}=={(2,2,3),(7,)}
print('N065: 所有本章断言通过')

N065: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释有负数时按和超标剪枝可能失效。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较可重复与不可重复选取。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 90. Subsets II（canonical）
- 47. Permutations II（canonical）
- 39. Combination Sum（canonical）
- 40. Combination Sum II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。